In [11]:
# IMPORTS
from rldb.utils import *
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os

In [2]:
# Load dataset
root = "/coc/cedarp-dxu345-0/datasets/RLDB/aria_hand_laundry_debug/lerobot_debug"
repo_id = "rpuns/aria_laundry_rl2"

episodes = [0, 1, 2]
dataset = RLDBDataset(repo_id=repo_id, root=root, local_files_only=True, episodes=episodes, mode="sample")


Returning existing local_dir `/coc/cedarp-dxu345-0/datasets/RLDB/aria_hand_laundry_debug/lerobot_debug` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/coc/cedarp-dxu345-0/datasets/RLDB/aria_hand_laundry_debug/lerobot_debug` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/coc/cedarp-dxu345-0/datasets/RLDB/aria_hand_laundry_debug/lerobot_debug` as remote repo cannot be accessed in `snapshot_download` (None).
Generating train split: 3000 examples [00:14, 200.54 examples/s]


In [4]:
# Get metadata
print(dataset.meta.info["features"])

image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"

{'observations.state.ee_pose': {'dtype': 'float64', 'shape': (6,), 'names': ['dim_0']}, 'observations.images.front_img_1': {'dtype': 'image', 'shape': (3, 240, 320), 'names': ['channel', 'height', 'width']}, 'actions_cartesian': {'dtype': 'prestacked_float64', 'shape': (100, 6), 'names': ['chunk_length', 'action_dim']}, 'metadata.embodiment': {'dtype': 'int32', 'shape': (1,), 'names': ['dim_0']}, 'timestamp': {'dtype': 'float32', 'shape': (1,), 'names': None}, 'frame_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'episode_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'task_index': {'dtype': 'int64', 'shape': (1,), 'names': None}}


In [5]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset,
                                          batch_size=1,
                                          shuffle=False)

In [7]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key="ariaJul29")

In [15]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(ims.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        arm = "right" if actions.shape[-1] == 7 or actions.shape[-1] == 3 else "both"
        ims[b] = draw_actions(ims[b], ac_type, "Purples", actions[b], extrinsics, intrinsics, arm=arm)

    return ims


In [23]:
save_dir = "./visualization/"
os.makedirs(save_dir, exist_ok=True)

num_batches = 1

for i, data in enumerate(data_loader):
    if i > num_batches:
        break
    ims = (data[image_key].permute(0, 2, 3, 1).cpu().numpy()).astype(np.uint8)
    actions = data[actions_key].cpu().numpy()

    ims_viz = visualize_actions(ims, actions, camera_transforms.extrinsics, camera_transforms.intrinsics)

    for j, im in enumerate(ims_viz):
        img_tensor = torch.from_numpy(im).permute(2, 0, 1)
        save_path = os.path.join(save_dir, f"image_{i}_{j}.png")
        io.write_png(img_tensor, save_path)

    print(f"Saved batch {i} images to {save_dir}")

Saved batch 0 images to ./visualization/
Saved batch 1 images to ./visualization/
